# GeoPython 2026 — Día 2: La pila vectorial
## Shapely → Fiona → GeoPandas

**Curso de Análisis Espacial con Python**  
Gabinete de Formación del CSIC — Estación Biológica de Doñana, Sevilla

---

> **Entorno recomendado:** Local (entorno `geopython2026`)  
> En Colab: `!pip install geopandas fiona shapely`

## Contenido

1. La pila de librerías vectoriales
2. Shapely — geometrías en Python
3. Fiona — I/O vectorial sobre OGR
4. GeoPandas — análisis vectorial de alto nivel

---
## 1. La pila de librerías vectoriales

Antes de escribir código, conviene entender cómo se relacionan las librerías que vamos a usar.  
Esta jerarquía es uno de los conceptos más importantes del curso:

```
GDAL / OGR  ←  librería C/C++ (~1998). El motor de todo.
    │
    ├── Fiona  ←  wrapper Python de OGR para lectura/escritura (2011)
    │               (GeoPandas ≥ 1.0 usa PyOGRIO internamente)
    │
    └── Shapely  ←  geometrías Python sobre GEOS (2007 → v2.0 en 2022)
            │
            └── GeoPandas  ←  GeoDataFrame = Shapely + Pandas + I/O (2013)
```

**¿Por qué importa esto?**
- Cuando algo falla en GeoPandas, el error suele venir de Shapely o de GDAL.
- Saber qué capa hace qué te permite depurar y elegir la herramienta correcta.
- Fiona y Shapely son útiles solos cuando necesitas control fino; GeoPandas cuando quieres velocidad de desarrollo.

Empezamos por la base y subimos.

---
## 2. Shapely

Shapely implementa el modelo de geometría de la especificación **OGC Simple Features**.  
Trabaja con objetos en memoria, sin CRS: solo coordenadas.

> A partir de la versión 2.0 (2022), Shapely usa **GEOS** de forma vectorizada y es significativamente más rápido.
  GeoPandas ≥ 0.14 requiere Shapely ≥ 2.0.

In [ ]:
from shapely.geometry import (
    Point, MultiPoint,
    LineString, MultiLineString,
    Polygon, MultiPolygon,
    GeometryCollection
)
import shapely
print(f'Shapely {shapely.__version__}')

### 2.1 Geometrías básicas

In [ ]:
# PUNTO
p = Point(711956, 4116309)   # coordenadas UTM (x, y)
print(f'Tipo:  {p.geom_type}')
print(f'WKT:   {p.wkt}')
print(f'Coords: {list(p.coords)}')
print(f'x={p.x}, y={p.y}')
p

In [ ]:
# LÍNEA
linea = LineString([(0, 0), (5, 5), (10, 0), (15, 5)])
print(f'Tipo:     {linea.geom_type}')
print(f'Longitud: {linea.length:.2f}')
print(f'Bounds:   {linea.bounds}')   # (minx, miny, maxx, maxy)
linea

In [ ]:
# POLÍGONO: exterior + agujeros (holes)
exterior = [(0, 0), (10, 0), (10, 10), (0, 10)]
agujero  = [(2, 2), (2, 4), (4, 4), (4, 2)]

poligono_simple = Polygon(exterior)
poligono_agujero = Polygon(exterior, holes=[agujero])

print(f'Área sin agujero:  {poligono_simple.area}')
print(f'Área con agujero:  {poligono_agujero.area}')  # 100 - 4 = 96
print(f'Perímetro exterior: {poligono_simple.length}')
print(f'Centroide: {poligono_simple.centroid}')
poligono_agujero

In [ ]:
# MULTI-geometrías: agrupan varios objetos del mismo tipo
multipunto = MultiPoint([(0, 0), (5, 5), (10, 0)])
multipoligono = MultiPolygon([
    Polygon([(0, 0), (2, 0), (2, 2), (0, 2)]),
    Polygon([(5, 5), (7, 5), (7, 7), (5, 7)]),
])

print(f'Nº polígonos: {len(list(multipoligono.geoms))}')
print(f'Área total:   {multipoligono.area}')
multipoligono

In [ ]:
# Crear desde WKT o WKB (muy útil al leer de bases de datos)
from shapely import wkt, wkb

geom_wkt = wkt.loads('POLYGON ((0 0, 4 0, 4 4, 0 4, 0 0))')
print(geom_wkt.area)

# Serializar
print(geom_wkt.wkt)
print(geom_wkt.wkb.hex()[:30], '...')

### 2.2 Atributos geométricos

In [ ]:
# Atributos más útiles de un polígono
p = Polygon([(0,0),(10,0),(10,6),(0,6)])

print(f'area:      {p.area}')
print(f'length:    {p.length}')         # perímetro
print(f'bounds:    {p.bounds}')         # (minx, miny, maxx, maxy)
print(f'centroid:  {p.centroid}')
print(f'envelope:  {p.envelope}')       # MBR como polígono
print(f'is_valid:  {p.is_valid}')
print(f'is_empty:  {p.is_empty}')
print(f'geom_type: {p.geom_type}')

In [ ]:
# representative_point(): garantizado que cae dentro del polígono
# (el centroide puede caer fuera en formas cóncavas)
c_forma = Polygon([(0,0),(10,0),(10,1),(1,1),(1,9),(10,9),(10,10),(0,10)])

centroide = c_forma.centroid
repr_pt   = c_forma.representative_point()

print(f'Centroide dentro:    {c_forma.contains(centroide)}')
print(f'Repr. point dentro:  {c_forma.contains(repr_pt)}')

### 2.3 Operaciones geométricas

In [ ]:
# Buffer
punto = Point(5, 5)
circulo = punto.buffer(3)        # por defecto 16 segmentos
circulo_fino = punto.buffer(3, resolution=64)

print(f'Área teórica (π·r²): {3.14159 * 9:.4f}')
print(f'Área buffer 16 seg:  {circulo.area:.4f}')
print(f'Área buffer 64 seg:  {circulo_fino.area:.4f}')

In [ ]:
# Operaciones booleanas: intersección, unión, diferencia
A = Polygon([(0,0),(4,0),(4,4),(0,4)])
B = Polygon([(2,2),(6,2),(6,6),(2,6)])

interseccion       = A.intersection(B)
union              = A.union(B)
diferencia_A_B     = A.difference(B)
diferencia_simetrica = A.symmetric_difference(B)

print(f'A área:                 {A.area}')
print(f'B área:                 {B.area}')
print(f'Intersección área:      {interseccion.area}')
print(f'Unión área:             {union.area}')
print(f'Diferencia A-B área:    {diferencia_A_B.area}')
print(f'Dif. simétrica área:    {diferencia_simetrica.area}')

# Verificación: área(A) + área(B) = área(unión) + área(intersección)
print(f'Verificación: {A.area + B.area} == {union.area + interseccion.area}')

In [ ]:
# Otras operaciones útiles
poligono = Polygon([(0,0),(10,0),(8,5),(10,10),(0,10),(2,5)])

print(f'Convex hull:  {poligono.convex_hull}')
print(f'Área original: {poligono.area:.2f}')
print(f'Área hull:     {poligono.convex_hull.area:.2f}')

# simplify: reduce vértices manteniendo forma aproximada
linea_compleja = LineString([(i, i + 0.05*(i%3)) for i in range(100)])
linea_simple   = linea_compleja.simplify(tolerance=0.5)
print(f'\nVértices original:   {len(list(linea_compleja.coords))}')
print(f'Vértices simplificada: {len(list(linea_simple.coords))}')

### 2.4 Predicados — relaciones topológicas

| Predicado | Significado |
|-----------|-------------|
| `intersects` | Se solapan en al menos un punto |
| `contains` | A contiene completamente a B |
| `within` | B está dentro de A (inverso de contains) |
| `touches` | Comparten borde pero no interior |
| `crosses` | Se cruzan (sin contener) |
| `disjoint` | No tienen ningún punto en común |
| `covers` | A cubre a B (incluye el borde) |
| `equals` | Geométricamente idénticos |

In [ ]:
A = Polygon([(0,0),(6,0),(6,6),(0,6)])
B = Polygon([(4,4),(8,4),(8,8),(4,8)])  # se solapa con A
C = Polygon([(1,1),(3,1),(3,3),(1,3)])  # dentro de A
D = Polygon([(7,7),(9,7),(9,9),(7,9)])  # fuera de A
borde = LineString([(6,0),(6,6)])       # en el borde de A

print('A vs B (solapado):')
print(f'  intersects: {A.intersects(B)} | contains: {A.contains(B)} | within: {B.within(A)}')

print('A vs C (C dentro de A):')
print(f'  contains: {A.contains(C)} | within: {C.within(A)} | disjoint: {A.disjoint(C)}')

print('A vs D (disjoint):')
print(f'  disjoint: {A.disjoint(D)} | intersects: {A.intersects(D)}')

print('A vs borde (touches):')
print(f'  touches: {A.touches(borde)} | intersects: {A.intersects(borde)}')

### 2.5 Shapely 2.0: operaciones vectorizadas

Shapely 2.0 permite operar sobre **arrays de geometrías** de forma eficiente, sin bucles Python.  
GeoPandas usa esto internamente, pero también puedes usarlo directamente.

In [ ]:
import numpy as np
import shapely

# Crear 1000 puntos de una vez (sin bucle)
x = np.random.uniform(0, 100, 1000)
y = np.random.uniform(0, 100, 1000)
puntos = shapely.points(x, y)   # array de geometrías

print(f'Tipo: {type(puntos)}, shape: {puntos.shape}')

# Buffer vectorizado: un buffer por punto, todo a la vez
buffers = shapely.buffer(puntos, radius=5)
print(f'Áreas (media): {shapely.area(buffers).mean():.4f}')  # ≈ π·25

# Contener: ¿qué puntos caen dentro de un polígono?
zona = Polygon([(25, 25), (75, 25), (75, 75), (25, 75)])
dentro = shapely.within(puntos, zona)
print(f'Puntos dentro de la zona: {dentro.sum()} de {len(puntos)}')

---
## 3. Fiona

Fiona es el wrapper Python de la capa **OGR** de GDAL.  
Su filosofía es simple: los archivos vectoriales son colecciones de **features**,  
y cada feature es un diccionario GeoJSON.

**¿Cuándo usar Fiona directamente?**
- Cuando necesitas leer/escribir archivos con control fino del schema.
- Cuando procesas features una a una sin cargar todo en memoria.
- Cuando quieres convertir formatos sin depender de GeoPandas.
- Para entender qué hace GeoPandas por debajo.

In [ ]:
import fiona
print(f'Fiona {fiona.__version__}')
print(f'Drivers disponibles: {len(fiona.supported_drivers)}')

# Drivers más comunes
comunes = ['ESRI Shapefile', 'GPKG', 'GeoJSON', 'OpenFileGDB', 'FlatGeobuf']
for d in comunes:
    modo = fiona.supported_drivers.get(d, 'no disponible')
    print(f'  {d:25s}: {modo}')

### 3.1 Abrir y explorar un archivo

In [ ]:
ruta_fincas = '../../data/fincas_ETRS89.shp'

with fiona.open(ruta_fincas) as src:
    print(f'Driver:      {src.driver}')
    print(f'CRS:         {src.crs}')
    print(f'EPSG:        {src.crs.to_epsg()}')
    print(f'Nº features: {len(src)}')
    print(f'Bounds:      {src.bounds}')
    print(f'\nSchema:')
    for k, v in src.schema.items():
        print(f'  {k}: {v}')

In [ ]:
# Cada feature es un diccionario tipo GeoJSON
with fiona.open(ruta_fincas) as src:
    feature = next(iter(src))   # primera feature

print('Claves:', list(feature.keys()))
print('\nAtributos (properties):')
for k, v in feature['properties'].items():
    print(f'  {k}: {v}')
print('\nGeometría (tipo):', feature['geometry']['type'])
print('Primer anillo (5 primeras coords):', feature['geometry']['coordinates'][0][:5])

In [ ]:
# Iterar sobre features con filtrado
from shapely.geometry import shape

fincas_privadas = []

with fiona.open(ruta_fincas) as src:
    for feature in src:
        if feature['properties']['TITULARIDA'] == 'Privada':
            geom = shape(feature['geometry'])   # dict → geometría Shapely
            fincas_privadas.append({
                'nombre':   feature['properties']['FINCA'],
                'hectareas': feature['properties']['Hectareas'],
                'geom':     geom
            })

print(f'Fincas privadas: {len(fincas_privadas)}')
for f in sorted(fincas_privadas, key=lambda x: x['hectareas'], reverse=True)[:5]:
    print(f"  {f['nombre']:30s} {f['hectareas']:.1f} ha")

### 3.2 Escribir un archivo vectorial con Fiona

In [ ]:
import os

# Escribir solo las fincas privadas a un nuevo shapefile
ruta_salida = '/tmp/fincas_privadas.shp'

with fiona.open(ruta_fincas) as src:
    schema_original = src.schema
    crs_original    = src.crs

    with fiona.open(ruta_salida, 'w',
                    driver='ESRI Shapefile',
                    schema=schema_original,
                    crs=crs_original) as dst:

        for feature in src:
            if feature['properties']['TITULARIDA'] == 'Privada':
                dst.write(feature)

# Verificar
with fiona.open(ruta_salida) as f:
    print(f'Features escritas: {len(f)}')
    print(f'CRS: {f.crs.to_epsg()}')

In [ ]:
# Crear un shapefile desde cero con schema personalizado
from shapely.geometry import mapping   # geometría Shapely → dict GeoJSON

schema_nuevo = {
    'geometry': 'Point',
    'properties': {
        'id':     'int',
        'nombre': 'str',
        'ndvi':   'float'
    }
}

puntos_muestreo = [
    (1, 'P01', Point(711000, 4116000), 0.65),
    (2, 'P02', Point(715000, 4112000), 0.42),
    (3, 'P03', Point(720000, 4118000), 0.78),
]

from fiona.crs import from_epsg

with fiona.open('/tmp/puntos_muestreo.shp', 'w',
                driver='ESRI Shapefile',
                schema=schema_nuevo,
                crs=from_epsg(25829)) as dst:

    for id_, nombre, geom, ndvi in puntos_muestreo:
        dst.write({
            'geometry':   mapping(geom),
            'properties': {'id': id_, 'nombre': nombre, 'ndvi': ndvi}
        })

print('Shapefile de puntos creado')

---
## 4. GeoPandas

GeoPandas extiende Pandas con una columna especial de geometrías (`geometry`).  
Un `GeoDataFrame` es esencialmente un DataFrame donde cada fila tiene:
- Sus atributos alfanuméricos (columnas normales de Pandas)
- Una geometría Shapely

Internamente usa Shapely para las operaciones y GDAL/PyOGRIO para la I/O.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
print(f'GeoPandas {gpd.__version__}')

### 4.1 Leer y escribir archivos

In [ ]:
# read_file() soporta shapefile, GeoPackage, GeoJSON, FlatGeobuf...
fincas = gpd.read_file('../../data/fincas_ETRS89.shp')
print(type(fincas))
print(fincas.shape)
fincas.head()

In [ ]:
# La columna 'geometry' contiene objetos Shapely
print(type(fincas['geometry'][0]))     # shapely.geometry.Polygon
print(fincas['geometry'].geom_type.value_counts())
print(f'\nCRS: {fincas.crs}')
print(f'EPSG: {fincas.crs.to_epsg()}')
print(f'Bounds:\n{fincas.total_bounds}')  # [minx, miny, maxx, maxy]

In [ ]:
# Acceso a atributos de Shapely desde la columna geometry
fincas['area_ha']   = fincas.geometry.area / 10000     # m² → ha
fincas['perimetro'] = fincas.geometry.length
fincas['centroide'] = fincas.geometry.centroid

print(fincas[['FINCA', 'TITULARIDA', 'Hectareas', 'area_ha']].head(8).to_string())

In [ ]:
# Escribir a distintos formatos
fincas.to_file('/tmp/fincas.gpkg', driver='GPKG', layer='fincas')
fincas.to_file('/tmp/fincas.geojson', driver='GeoJSON')

# Leer el GeoPackage
fincas_gpkg = gpd.read_file('/tmp/fincas.gpkg', layer='fincas')
print(f'Leídas desde GPKG: {len(fincas_gpkg)} features')

### 4.2 CRS y reproyecciones

El CRS (Coordinate Reference System) es fundamental en análisis espacial.  
GeoPandas usa **pyproj** internamente para gestionar proyecciones.

In [ ]:
# Ver el CRS completo
print(fincas.crs)
print(f'\nProj4: {fincas.crs.to_proj4()}')
print(f'WKT (resumido): {fincas.crs.name}')
print(f'Es geográfico: {fincas.crs.is_geographic}')
print(f'Es proyectado: {fincas.crs.is_projected}')
print(f'Unidades: {fincas.crs.axis_info[0].unit_name}')

In [ ]:
# Reproyectar: to_crs()
fincas_geo  = fincas.to_crs(epsg=4326)   # WGS84 geográfico
fincas_etrs = fincas.to_crs(epsg=25830)  # ETRS89 UTM zona 30N

print('Original (EPSG:25829):')
print(f'  {fincas.geometry[0].centroid}')

print('En WGS84 (EPSG:4326):')
print(f'  {fincas_geo.geometry[0].centroid}')

print('En UTM 30N (EPSG:25830):')
print(f'  {fincas_etrs.geometry[0].centroid}')

In [ ]:
# ¡Siempre en el mismo CRS antes de operar!
# Esto fallará (o dará resultados sin sentido):
islas = gpd.read_file('../../data/CanaryIslands_exploded_etrs.shp')
print(f'Fincas CRS:  {fincas.crs.to_epsg()}')
print(f'Islas CRS:   {islas.crs.to_epsg()}')

# Solución: reproyectar a un CRS común
islas_utm = islas.to_crs(epsg=25830)
fincas_utm = fincas.to_crs(epsg=25830)
print('\nAmbas en EPSG:25830 ✓')

### 4.3 Selección y filtrado

In [ ]:
# Filtrado atributual (igual que Pandas)
publicas  = fincas[fincas['TITULARIDA'] == 'Pública']
privadas  = fincas[fincas['TITULARIDA'] == 'Privada']
grandes   = fincas[fincas['Hectareas'] > 500]

print(f'Públicas:  {len(publicas)}')
print(f'Privadas:  {len(privadas)}')
print(f'> 500 ha:  {len(grandes)}')

In [ ]:
# Filtrado espacial con cx (coordinate indexer)
# Seleccionar features dentro de un bounding box
minx, miny, maxx, maxy = 710000, 4110000, 720000, 4120000
fincas_bbox = fincas.cx[minx:maxx, miny:maxy]
print(f'Fincas en el bbox: {len(fincas_bbox)}')

In [ ]:
# Consulta espacial con clip (recortar al contorno exacto)
# Creamos un polígono de recorte
from shapely.geometry import box

rectangulo = gpd.GeoDataFrame(
    geometry=[box(710000, 4110000, 720000, 4120000)],
    crs=fincas.crs
)

fincas_clip = gpd.clip(fincas, rectangulo)
print(f'Features recortadas: {len(fincas_clip)}')
print(f'Área total recortada: {fincas_clip.geometry.area.sum()/10000:.1f} ha')

### 4.4 Operaciones espaciales

In [ ]:
# Buffer colectivo sobre todo el GeoDataFrame
fincas_buf = fincas.copy()
fincas_buf['geometry'] = fincas.geometry.buffer(100)  # 100 metros

print(f'Área media original: {fincas.geometry.area.mean()/10000:.1f} ha')
print(f'Área media buffered: {fincas_buf.geometry.area.mean()/10000:.1f} ha')

In [ ]:
# Dissolve: fusionar features por atributo
# Agrupa todas las fincas por titularidad y las fusiona
titularidad = fincas.dissolve(by='TITULARIDA', aggfunc={
    'Hectareas': 'sum',
    'FINCA': 'count'
}).rename(columns={'FINCA': 'n_fincas'})

print(titularidad[['Hectareas', 'n_fincas']])
titularidad.geometry.plot()

In [ ]:
# Overlay: operaciones entre dos GeoDataFrames
# intersection, union, difference, symmetric_difference, identity

# Crear dos capas de ejemplo
zona_norte = gpd.GeoDataFrame(
    geometry=[box(700000, 4115000, 740000, 4130000)], crs=fincas.crs)
zona_sur   = gpd.GeoDataFrame(
    geometry=[box(700000, 4100000, 740000, 4118000)], crs=fincas.crs)

fincas_norte = gpd.overlay(fincas, zona_norte, how='intersection')
fincas_sur   = gpd.overlay(fincas, zona_sur,   how='intersection')

print(f'Fincas (parte) en zona norte: {len(fincas_norte)}')
print(f'Fincas (parte) en zona sur:   {len(fincas_sur)}')

### 4.5 Joins

In [ ]:
# JOIN ATRIBUTUAL: como un merge de Pandas
# Añadir información extra a las fincas
info_extra = pd.DataFrame({
    'FINCA': ['ROCINA', 'ACEBUCHE', 'MADRE', 'MARISMILLAS'],
    'habitat': ['Marisma', 'Matorral mediterráneo', 'Corredor fluvial', 'Marisma'],
    'gestion': ['RENPA', 'RENPA', 'RENPA', 'RENPA']
})

fincas_info = fincas.merge(info_extra, on='FINCA', how='left')
print(fincas_info[['FINCA', 'TITULARIDA', 'habitat']].dropna().head())

In [ ]:
# SPATIAL JOIN: unir por relación espacial
# ¿Qué municipio contiene cada finca?
municipios = gpd.read_file('../../data/13_01_TerminoMunicipal.shp')
fincas_muni = fincas.to_crs(municipios.crs)  # mismo CRS

print(f'Columnas municipios: {municipios.columns.tolist()}')

# predicate: 'intersects', 'within', 'contains', 'touches'
fincas_con_muni = gpd.sjoin(
    fincas_muni, municipios,
    how='left',
    predicate='intersects'
)
print(f'\nResultado sjoin: {len(fincas_con_muni)} filas')
print(fincas_con_muni.columns.tolist())

### 4.6 Visualización básica

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Coloreado por atributo categórico
fincas.plot(ax=axes[0],
            column='TITULARIDA',
            categorical=True,
            legend=True,
            cmap='Set2',
            edgecolor='black',
            linewidth=0.5)
axes[0].set_title('Titularidad')

# 2. Coloreado por atributo numérico (choropleth)
fincas.plot(ax=axes[1],
            column='Hectareas',
            legend=True,
            cmap='YlOrRd',
            edgecolor='black',
            linewidth=0.5)
axes[1].set_title('Superficie (ha)')

# 3. Capas superpuestas
fincas[fincas['TITULARIDA']=='Pública'].plot(
    ax=axes[2], color='steelblue', alpha=0.5, edgecolor='navy', linewidth=0.5, label='Pública')
fincas[fincas['TITULARIDA']=='Privada'].plot(
    ax=axes[2], color='salmon', alpha=0.5, edgecolor='darkred', linewidth=0.5, label='Privada')
axes[2].legend()
axes[2].set_title('Pública vs Privada')

for ax in axes:
    ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Explore: mapa interactivo (Folium) con una sola línea
# (Funciona en Jupyter, no en algunos entornos de VS Code)
fincas.explore(
    column='TITULARIDA',
    cmap='Set2',
    tooltip=['FINCA', 'TITULARIDA', 'Hectareas'],
    tiles='CartoDB positron'
)

---
## Resumen del día 2

| Librería | Lo que hace | Cuándo usarla |
|----------|-------------|---------------|
| **Shapely** | Geometrías y operaciones espaciales en memoria | Siempre que necesites geometría sin I/O |
| **Fiona** | Leer/escribir archivos vectoriales (sobre OGR) | Control fino de I/O, procesar sin cargar todo |
| **GeoPandas** | Análisis vectorial completo (Shapely + Pandas + I/O) | El día a día |

**Flujo habitual:**
```python
gdf = gpd.read_file('datos.shp')          # Fiona por debajo
gdf = gdf.to_crs(epsg=25830)              # pyproj por debajo
gdf['buffer'] = gdf.geometry.buffer(100)  # Shapely por debajo
resultado = gpd.sjoin(gdf, otra)          # Shapely por debajo
resultado.to_file('salida.gpkg')          # Fiona por debajo
```

**Mañana (Día 3):** GDAL/OGR → Numpy → Rasterio. La misma idea pero para datos raster.

---
*GeoPython 2026 — CSIC / Estación Biológica de Doñana*